# Cuadernillo de preparación de datos - Bike Sharing (Kaggle)

Este cuadernillo integra y prepara los datasets `train.csv` y `test.csv` del problema **Bike Sharing Demand**. 

## Contexto general
El objetivo del problema es modelar y predecir la demanda de bicicletas compartidas según variables de tiempo y clima. 

- `train.csv` incluye las variables explicativas y las variables objetivo (`casual`, `registered`, `count`).
- `test.csv` incluye solo las variables explicativas (sin objetivos), para generar predicciones.

Este notebook está diseñado para que luego puedas reutilizarlo en ejercicios de:
- **Regresión** (predicción de una cantidad continua, por ejemplo `count`).
- **Clasificación** (crear clases de demanda: baja, media, alta).
- **Redes neuronales** (insumos totalmente numéricos y escalados).


In [ ]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 140)


In [ ]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print('Train shape:', train.shape)
print('Test shape :', test.shape)

display(train.head(3))
display(test.head(3))


## Diccionario de columnas - `train.csv`

| Columna | Tipo base | Significado |
|---|---|---|
| `datetime` | texto/fecha | Marca temporal del registro (año-mes-día hora). |
| `season` | categórica codificada | Estación del año (1: primavera, 2: verano, 3: otoño, 4: invierno). |
| `holiday` | binaria | 1 si es feriado, 0 si no lo es. |
| `workingday` | binaria | 1 si es día laborable (no fin de semana ni feriado), 0 en caso contrario. |
| `weather` | categórica ordinal | Condición meteorológica (mejor a peor clima por categoría). |
| `temp` | numérica continua | Temperatura real en grados Celsius. |
| `atemp` | numérica continua | Temperatura percibida en grados Celsius. |
| `humidity` | numérica discreta | Humedad relativa (%). |
| `windspeed` | numérica continua | Velocidad del viento. |
| `casual` | objetivo numérico | Usuarios no registrados que alquilan bicicletas. |
| `registered` | objetivo numérico | Usuarios registrados que alquilan bicicletas. |
| `count` | objetivo numérico | Total de alquileres (`casual + registered`). |


## Diccionario de columnas - `test.csv`

| Columna | Tipo base | Significado |
|---|---|---|
| `datetime` | texto/fecha | Marca temporal del registro (año-mes-día hora). |
| `season` | categórica codificada | Estación del año. |
| `holiday` | binaria | Indicador de feriado. |
| `workingday` | binaria | Indicador de día laborable. |
| `weather` | categórica ordinal | Estado del clima. |
| `temp` | numérica continua | Temperatura real. |
| `atemp` | numérica continua | Temperatura percibida. |
| `humidity` | numérica discreta | Humedad relativa. |
| `windspeed` | numérica continua | Velocidad del viento. |


## Análisis inicial de calidad de datos

Verificamos tipos, nulos y estadísticas para entender qué transformación aplicar después.

In [ ]:
print('--- Tipos de train ---')
print(train.dtypes)
print('\n--- Tipos de test ---')
print(test.dtypes)

print('\n--- Nulos en train ---')
print(train.isna().sum())
print('\n--- Nulos en test ---')
print(test.isna().sum())

display(train.describe(include='all').T)


## Transformaciones propuestas y justificación

A continuación convertimos el dataset para que sea 100% utilizable en modelos de ML.

1. **Parseo de `datetime` a fecha**:
   - Motivo: como texto no aporta estructura temporal al modelo.
   - Cambio: convertir a `datetime64`.

2. **Ingeniería temporal** (`year`, `month`, `day`, `hour`, `dayofweek`, `is_weekend`):
   - Motivo: la demanda depende mucho de la hora, día y estacionalidad.
   - Cambio: crear variables numéricas derivadas.

3. **Codificación cíclica de hora y mes** (`sin`/`cos`):
   - Motivo: la hora 23 y la hora 0 están "cerca", y el modelo lineal no lo captura sin codificación cíclica.
   - Cambio: agregar `hour_sin`, `hour_cos`, `month_sin`, `month_cos`.

4. **Tratamiento de categóricas** (`season`, `weather`):
   - Motivo: son categorías, no magnitudes continuas.
   - Cambio: One-Hot Encoding para pasar a columnas numéricas binarias.

5. **Escalado de numéricas**:
   - Motivo: ayuda a modelos sensibles a escala (Regresión con regularización, SVM, redes neuronales).
   - Cambio: `StandardScaler`.

6. **Definición de objetivo para clasificación**:
   - Motivo: `count` es continuo (regresión). Para clasificación se crea una etiqueta por tramos.
   - Cambio: variable `demand_class` (baja/media/alta) con cuantiles.


In [ ]:
def add_time_features(df):
    df = df.copy()
    df['datetime'] = pd.to_datetime(df['datetime'])
    df['year'] = df['datetime'].dt.year
    df['month'] = df['datetime'].dt.month
    df['day'] = df['datetime'].dt.day
    df['hour'] = df['datetime'].dt.hour
    df['dayofweek'] = df['datetime'].dt.dayofweek
    df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)

    # Codificación cíclica para variables periódicas
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    return df

train_feat = add_time_features(train)
test_feat = add_time_features(test)

display(train_feat.head(2))


In [ ]:
# Variables objetivo (regresión)
y_count = train_feat['count'].copy()
y_casual = train_feat['casual'].copy()
y_registered = train_feat['registered'].copy()

# Variables objetivo (clasificación por cuantiles)
y_class = pd.qcut(y_count, q=3, labels=['baja', 'media', 'alta'])
train_feat['demand_class'] = y_class

# Matriz de features para entrenar (sin targets ni datetime original)
drop_targets = ['datetime', 'casual', 'registered', 'count', 'demand_class']
X_train_raw = train_feat.drop(columns=drop_targets)
X_test_raw = test_feat.drop(columns=['datetime'])

print('X_train_raw:', X_train_raw.shape)
print('X_test_raw :', X_test_raw.shape)
print('\nDistribucion de clases:')
print(train_feat['demand_class'].value_counts())


In [ ]:
# Definimos columnas categóricas y numéricas
categorical_features = ['season', 'weather']
numeric_features = [c for c in X_train_raw.columns if c not in categorical_features]

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

X_train_processed = preprocessor.fit_transform(X_train_raw)
X_test_processed = preprocessor.transform(X_test_raw)

cat_names = preprocessor.named_transformers_['cat']['onehot'].get_feature_names_out(categorical_features)
all_feature_names = np.array(numeric_features + list(cat_names))

X_train_processed_df = pd.DataFrame(X_train_processed, columns=all_feature_names, index=X_train_raw.index)
X_test_processed_df = pd.DataFrame(X_test_processed, columns=all_feature_names, index=X_test_raw.index)

print('X_train_processed:', X_train_processed_df.shape)
print('X_test_processed :', X_test_processed_df.shape)
display(X_train_processed_df.head(3))


## Resultado final para modelado

Después del preprocesamiento tienes:

- `X_train_processed_df` y `X_test_processed_df`: matrices numéricas listas para entrenar/predicción.
- `y_count`, `y_casual`, `y_registered`: objetivos para **regresión**.
- `y_class`: objetivo para **clasificación**.

Esto ya es compatible con modelos de `scikit-learn`, XGBoost/LightGBM y redes neuronales (TensorFlow/PyTorch).


In [ ]:
# Export opcional de datasets procesados
X_train_processed_df.to_csv('X_train_processed.csv', index=False)
X_test_processed_df.to_csv('X_test_processed.csv', index=False)

pd.DataFrame({
    'y_count': y_count,
    'y_casual': y_casual,
    'y_registered': y_registered,
    'y_class': y_class.astype(str)
}).to_csv('y_train_targets.csv', index=False)

print('Archivos exportados: X_train_processed.csv, X_test_processed.csv, y_train_targets.csv')
